In [5]:
#提取nc文件中albedo和其他检查操作
import xarray as xr
import numpy as np
import pandas as pd

# 1. 打开nc文件
nc_file = r"F:\TUB\greenlandProject\ERA5LAND\forecastAlbedo.nc"
ds = xr.open_dataset(nc_file)

# 2. 查看数据集的基本信息（可选）
print(ds)

# 3. 假设你的反照率变量名为“albedo”
albedo_data = ds["fal"]

# 4. 获取坐标信息
lat = ds["latitude"]
lon = ds["longitude"]
valid_time = ds["valid_time"]

# 5. 如果需要，可以进一步筛选时间范围、经纬度范围等。例如：
#    这里举例把时间限制在 1950-2020（如果 ds 已经是 1950-2020 就不需要）
albedo_data = albedo_data.sel(valid_time=slice("1950-01-01","2020-12-01"))

# 6. 如果需要，可以检查数据统计量，例如：
print(albedo_data)
print(albedo_data.mean(dim=["valid_time","latitude","longitude"]))  # 全局平均（举例）


<xarray.Dataset>
Dimensions:     (valid_time: 900, latitude: 401, longitude: 651)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 1950-01-01 1950-02-01 ... 2024-12-01
  * latitude    (latitude) float64 90.0 89.9 89.8 89.7 ... 50.3 50.2 50.1 50.0
  * longitude   (longitude) float64 -75.0 -74.9 -74.8 ... -10.2 -10.1 -10.0
Data variables:
    number      int64 ...
    expver      (valid_time) object ...
    asn         (valid_time, latitude, longitude) float32 ...
    fal         (valid_time, latitude, longitude) float32 ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
<xarray.DataArray 'fal' (valid_time: 852, latitude: 401, longitude: 651)>
[222415452 values with dtype=float32]
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 1950-01-01 1950-02-01

In [19]:
#检查，读取csv文件头看是否损坏
import io
import pandas as pd

csv_file = r"F:\TUB\greenlandProject\document\dataverse_files\dataverse_files\month\CEN1_month.csv"

# 先用 Python 内置 open() 读取字节, 试着强制 decode 并替换无法识别的字符
try:
    with open(csv_file, 'r', encoding='utf-8', errors='replace') as f:
        raw_text = f.read()
except Exception as e:
    print(f"无法以 UTF-8 even with 'replace' 打开文件: {e}")
    raw_text = None

if raw_text:
    # 打印前几百个字符，看看是乱码还是正常文本
    print(raw_text[:500])
    
    # 尝试用 Pandas 解析
    df = pd.read_csv(io.StringIO(raw_text), sep=',', engine='python', on_bad_lines='skip')
    print(df.head())


PK    ! ��yh  
  [Content_Types].xml �(�                                                                                                                                                                                                                                                                                                                                                                                                                                                            
                              PK    ! ��yh  
0    [Content_Types].xml �(�             ...
1                                     �{\t����]�[M�
2  ����*l��׆����~Q�q�FKP^.m�@�!�P��5U^++���>�...
3  �p/ă �P�T�[V��$���(����%Q0���$%Y�S���rl۳J�8...
4  RU�ԙ*�Q�ZM3�h�C�%}�p�!�*jX��!v<do�|!a �E...


In [1]:
# 一个csv文件的提取time和albedo操作
import pandas as pd

# 定义文件路径
input_file = r'F:\\TUB\\greenlandProject\\NEWdataverse_files\\month\\CEN_month.csv'
output_file = r'F:\\TUB\\greenlandProject\\NEWdataverse_files\\month\\CEN_monthalbedo.csv'

# 读取数据
try:
    data = pd.read_csv(input_file)
    
    # 提取 time 和 albedo 列
    extracted_data = data[['time', 'albedo']]
    
    # 保存为新 CSV 文件
    extracted_data.to_csv(output_file, index=False)
    print(f"文件已成功保存为 {output_file}")
except FileNotFoundError:
    print(f"未找到输入文件: {input_file}")
except KeyError as e:
    print(f"文件中缺少指定的列: {e}")
except Exception as e:
    print(f"发生错误: {e}")


文件已成功保存为 F:\\TUB\\greenlandProject\\NEWdataverse_files\\month\\CEN_monthalbedo.csv


In [2]:
# 整个文件路径下包含time和albedo的csv合并成一个csv操作
import os
import pandas as pd

# 定义文件夹路径
input_folder = r'F:\\TUB\\greenlandProject\\NEWdataverse_files\\month'
output_folder = input_folder  # 输出文件保存在相同文件夹

# 遍历文件夹中的所有文件
for file_name in os.listdir(input_folder):
    # 只处理 CSV 文件
    if file_name.endswith('.csv'):
        input_file = os.path.join(input_folder, file_name)
        
        try:
            # 读取数据
            data = pd.read_csv(input_file)
            
            # 检查是否包含 time 和 albedo 列
            if 'time' in data.columns and 'albedo' in data.columns:
                # 提取 time 和 albedo 列
                extracted_data = data[['time', 'albedo']]
                
                # 生成输出文件名
                output_file_name = file_name.replace('.csv', 'albedo.csv')
                output_file = os.path.join(output_folder, output_file_name)
                
                # 保存为新 CSV 文件
                extracted_data.to_csv(output_file, index=False)
                print(f"成功处理文件: {file_name} -> {output_file_name}")
            else:
                print(f"文件 {file_name} 缺少 time 或 albedo 列，已跳过。")
        except Exception as e:
            print(f"处理文件 {file_name} 时发生错误: {e}")


处理文件 CEN_month.csv 时发生错误: [Errno 13] Permission denied: 'F:\\\\TUB\\\\greenlandProject\\\\NEWdataverse_files\\\\month\\CEN_monthalbedo.csv'
成功处理文件: CEN_monthalbedo.csv -> CEN_monthalbedoalbedo.csv
成功处理文件: CP1_month.csv -> CP1_monthalbedo.csv
成功处理文件: DY2_month.csv -> DY2_monthalbedo.csv
成功处理文件: EGP_month.csv -> EGP_monthalbedo.csv
成功处理文件: FRE_month.csv -> FRE_monthalbedo.csv
成功处理文件: HUM_month.csv -> HUM_monthalbedo.csv
成功处理文件: JAR_month.csv -> JAR_monthalbedo.csv
成功处理文件: KAN_B_month.csv -> KAN_B_monthalbedo.csv
成功处理文件: KAN_L_month.csv -> KAN_L_monthalbedo.csv
成功处理文件: KAN_M_month.csv -> KAN_M_monthalbedo.csv
成功处理文件: KAN_T_month.csv -> KAN_T_monthalbedo.csv
成功处理文件: KAN_U_month.csv -> KAN_U_monthalbedo.csv
成功处理文件: KPC_L_month.csv -> KPC_L_monthalbedo.csv
成功处理文件: KPC_U_month.csv -> KPC_U_monthalbedo.csv
成功处理文件: LYN_L_month.csv -> LYN_L_monthalbedo.csv
成功处理文件: LYN_T_month.csv -> LYN_T_monthalbedo.csv
成功处理文件: MIT_month.csv -> MIT_monthalbedo.csv
成功处理文件: NAE_month.csv -> NAE_monthalbedo.csv
成功

In [3]:
import os
import pandas as pd

# 定义文件夹路径
input_folder = r'F:\\TUB\\greenlandProject\\NEWdataverse_files\\month'
output_file = os.path.join(input_folder, 'combined_albedo.csv')

# 初始化空的 DataFrame 用于存储合并数据
combined_data = pd.DataFrame()

# 遍历文件夹中的所有文件
for file_name in os.listdir(input_folder):
    # 只处理以 albedo.csv 结尾的文件
    if file_name.endswith('albedo.csv'):
        input_file = os.path.join(input_folder, file_name)
        
        try:
            # 读取数据
            data = pd.read_csv(input_file)
            
            # 检查是否包含 time 和 albedo 列
            if 'time' in data.columns and 'albedo' in data.columns:
                # 提取 time 和 albedo 列
                data = data[['time', 'albedo']]
                
                # 添加站点名称列
                station_name = file_name.replace('albedo.csv', '')
                data['station'] = station_name
                
                # 追加到合并数据中
                combined_data = pd.concat([combined_data, data], ignore_index=True)
                print(f"成功合并文件: {file_name}")
            else:
                print(f"文件 {file_name} 缺少 time 或 albedo 列，已跳过。")
        except Exception as e:
            print(f"处理文件 {file_name} 时发生错误: {e}")

# 保存合并后的数据到输出文件
try:
    combined_data.to_csv(output_file, index=False)
    print(f"所有文件已成功合并为: {output_file}")
except Exception as e:
    print(f"保存合并文件时发生错误: {e}")


成功合并文件: CEN_monthalbedo.csv
成功合并文件: CP1_monthalbedo.csv
成功合并文件: DY2_monthalbedo.csv
成功合并文件: EGP_monthalbedo.csv
成功合并文件: FRE_monthalbedo.csv
成功合并文件: HUM_monthalbedo.csv
成功合并文件: JAR_monthalbedo.csv
成功合并文件: KAN_B_monthalbedo.csv
成功合并文件: KAN_L_monthalbedo.csv
成功合并文件: KAN_M_monthalbedo.csv
成功合并文件: KAN_T_monthalbedo.csv
成功合并文件: KAN_U_monthalbedo.csv
成功合并文件: KPC_L_monthalbedo.csv
成功合并文件: KPC_U_monthalbedo.csv
成功合并文件: LYN_L_monthalbedo.csv
成功合并文件: LYN_T_monthalbedo.csv
成功合并文件: MIT_monthalbedo.csv
成功合并文件: NAE_monthalbedo.csv
成功合并文件: NAU_monthalbedo.csv
成功合并文件: NEM_monthalbedo.csv
成功合并文件: NSE_monthalbedo.csv
成功合并文件: NUK_K_monthalbedo.csv
成功合并文件: NUK_L_monthalbedo.csv
成功合并文件: NUK_N_monthalbedo.csv
成功合并文件: NUK_U_monthalbedo.csv
成功合并文件: QAS_A_monthalbedo.csv
成功合并文件: QAS_L_monthalbedo.csv
成功合并文件: QAS_M_monthalbedo.csv
成功合并文件: QAS_U_monthalbedo.csv
成功合并文件: RED_L_monthalbedo.csv
成功合并文件: SCO_L_monthalbedo.csv
成功合并文件: SCO_U_monthalbedo.csv
成功合并文件: SDL_monthalbedo.csv
成功合并文件: SDM_monthalbedo.csv
成功合并文件: 

In [4]:
import os
import pandas as pd

# 定义文件路径
input_folder = r'F:\\TUB\\greenlandProject\\NEWdataverse_files\\month'
metadata_file = r'F:\\TUB\\greenlandProject\\NEWdataverse_files\\metadata\\AWS_sites_metadata.csv'
combined_file = os.path.join(input_folder, 'combined_albedo.csv')
output_file = os.path.join(input_folder, 'combined_albedo_with_metadata.csv')

try:
    # 读取 combined_albedo.csv
    combined_data = pd.read_csv(combined_file)

    # 读取 AWS_sites_metadata.csv
    metadata = pd.read_csv(metadata_file)

    # 检查必要列是否存在
    if 'site_id' not in metadata.columns:
        raise KeyError("AWS_sites_metadata.csv 文件中缺少 site_id 列")
    
    if 'station' not in combined_data.columns:
        raise KeyError("combined_albedo.csv 文件中缺少 station 列")

    # 合并数据，基于站点名匹配
    merged_data = pd.merge(
        combined_data, 
        metadata, 
        left_on='station', 
        right_on='site_id', 
        how='left'
    )

    # 保存合并后的文件
    merged_data.to_csv(output_file, index=False)
    print(f"文件已成功保存为 {output_file}")
except FileNotFoundError as e:
    print(f"文件未找到: {e}")
except KeyError as e:
    print(f"文件中缺少指定的列: {e}")
except Exception as e:
    print(f"发生错误: {e}")


发生错误: 'utf-8' codec can't decode byte 0xb8 in position 184285: invalid start byte
